In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%sql
create schema if not exists nutrifit_catalog.gold;

In [0]:
%sql
USE nutrifit_catalog.silver;

In [0]:
%sql
select * from silver.daily_nutrition
limit 5

In [0]:
%sql
select user_id, date,
    sum(calories) as calories_consumed,
    sum(protein_g) as protein_consumed,
    sum(carbs_g) as carbs_consumed,
    sum(fat_g) as fat_consumed,
    count(*) as meals_count
from silver.daily_nutrition
group by user_id, date
limit 5

In [0]:
nutrition_daily = (
    spark.table("nutrifit_catalog.silver.daily_nutrition")
    .groupBy("user_id", "date")
    .agg(
        F.sum("calories").alias("calories_consumed"),
        F.sum("protein_g").alias("protein_g"),
        F.sum("carbs_g").alias("carbs_g"),
        F.sum("fat_g").alias("fat_g"),
        F.count("*").alias("meals_count")
    )
)
display(nutrition_daily.limit(5))

In [0]:
daily_activity = spark.table("nutrifit_catalog.silver.daily_activity")
display(daily_activity.limit(5))

In [0]:
# rename activity date column to date for join
activity = (
    spark.table("nutrifit_catalog.silver.daily_activity")
    .withColumnRenamed("activity_date", "date")
)
display(activity.limit(5))

In [0]:
# create fact table and add has_activity column to indicate if user had activity on that day and calories_balance 
# have 48 days with meals but no activity, so in fact table I keep all days with nutrition data (left join)
fact_daily_summary = (
    nutrition_daily
    .join(activity, on = ["user_id", "date"], how = "left")
    .select(
        "user_id",
        "date",
        F.col("TotalSteps").alias("total_steps"),
        F.col("TotalDistance").alias("total_distance_km"),
        F.col("Calories").alias("calories_burned"),
        "calories_consumed",
        "protein_g",
        "carbs_g",
        "fat_g",
        "meals_count"
    )
    .withColumn("has_activity", F.col("calories_burned").isNotNull())
    .withColumn("calories_balance", F.col("calories_consumed") - F.col("calories_burned"))
)

display(fact_daily_summary.limit(5))
print(fact_daily_summary.count())

In [0]:
total = fact_daily_summary.count()
no_activity = fact_daily_summary.filter(~F.col("has_activity")).count()
print(f"Total rows: {total}, days without activity: {no_activity}")

In [0]:
fact_daily_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.gold.fact_daily_summary")

In [0]:
%sql
select count(*) from nutrifit_catalog.gold.fact_daily_summary

In [0]:
%sql
select count(*) from nutrifit_catalog.gold.fact_daily_summary where calories_balance > 0;

In [0]:
users = spark.table("nutrifit_catalog.silver.users")
display(users.limit(5))

In [0]:
# create dim table with users
dim_users = (
    users.select(
        "user_id",
        "name",
        "age",
        "weight_kg",
        "height_cm",
        "daily_step_goal",
        "daily_calorie_goal"
    )
    .withColumn("bmi", F.round(F.col("weight_kg") / ((F.col("height_cm") / 100) ** 2), 1))
)
display(dim_users.limit(5))
print(dim_users.count())

In [0]:
dim_users.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.gold.dim_users")

In [0]:
nutrition = spark.table("nutrifit_catalog.silver.daily_nutrition")
display(nutrition.limit(5))

In [0]:
# for dim_foods table, we have surrogate key
nutrition = spark.table("nutrifit_catalog.silver.daily_nutrition")

dim_foods = (
    nutrition
    .select(
        "food_name",
        "calories",
        "protein_g",
        "carbs_g",
        "fat_g"
    )
    .distinct()
    .withColumn("food_id", F.row_number().over(Window.orderBy("food_name")))
    .select(
        "food_id",
        "food_name",
        "calories",
        "protein_g",
        "carbs_g",
        "fat_g"
    )
)
display(dim_foods)
print(dim_foods.count())

In [0]:
dim_foods.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.gold.dim_foods")

In [0]:
dim_foods = spark.table("nutrifit_catalog.gold.dim_foods")

fact_meals = (
    nutrition
    .join(dim_foods.select("food_id", "food_name"), on = "food_name", how = "inner")
    .select("log_id", "user_id", "date", "meal_type", "food_id",
            "calories", "protein_g", "carbs_g", "fat_g")
)

display(fact_meals.limit(5))
print(fact_meals.count())

In [0]:
fact_meals.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.gold.fact_meals")